# 09 · Recursive un-starving & the new-hero ensemble

*Edge arc · 07 gate-liveness · 08 starvation · **09 the forward levers** — machinery: `edge_07`, `edge08_lib`*

`08` showed the gate is starved + subordinate, and that the one forward lever is **multi-task `r1`-aux
un-starving** (predict the pre-d8 residual as an aux head). This notebook lands the two results that turn
that into a deployable hero: (1) **un-starve the *anticipation* too** — gate the aux by d8's bite `|ĝ|`
(the un-starved conditioner), which ~doubles the gain and removes the fold reversal; and (2) the
**EBM ⊕ MTFM ensemble** — the two are diverse enough (corr ≈ 0.3) that averaging beats the EBM alone.
All run live on the local realrank cache.

In [1]:
import html, inspect, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "notebooks/results/edge_features"))

import resid_amortized as ra
from src.evaluation.metrics import apply_duan_smearing
from src.models.regime_moe import MultiTaskFM
from edge08_lib import MultiHeadFM
from xgboost import XGBRegressor
from interpret.glassbox import ExplainableBoostingRegressor as EBR

def _details(f, open_=False):
    mod = getattr(f, "__module__", "local") or "local"
    mod = (mod.replace("src.", "src/").replace(".", "/") + ".py") if "src" in mod else mod
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}</code></summary>\n\n{body}\n\n</details>"
def show_one(f): return Markdown(_details(f))

# shared data: realrank close residual r2 = (y - base) - d8(all hours); r1 = base residual
CID = "ebm_all_buckets_tw1000_enetreg2_realrank_rf480_slim"
c = ra.load_cache(CID)
tw, feats, Xs, y = c["cell"]["train_win"], c["feats"], c["Xs"], c["y"]
ridge, base = c["ridge_oos"], c["base"][tw:]
hour = Xs[tw:, feats.index("hour")]
r1a = (y[tw:] - ridge).astype("float32")
keep = [f for f in feats if f.startswith("har_ma_") or "cumrv" in f.lower() or f == "hour"]
Xall = Xs[tw:][:, [feats.index(f) for f in keep]].astype("float32")
g = XGBRegressor(max_depth=8, n_estimators=80, n_jobs=4).fit(Xall, r1a)
r2a = (r1a - g.predict(Xall)).astype("float32")
ci = np.where((hour >= 16) & (hour <= 19))[0]
X, r1, r2 = Xall[ci], r1a[ci], r2a[ci]
ridc, yc, bc, N = ridge[ci], y[tw:][ci], base[ci], len(ci)

def ql(p, te):
    pr, trr = apply_duan_smearing(ridc[te] + p, yc[te], bc[te]); m = (trr > 0) & (pr > 0); rr = trr[m] / pr[m]
    return float(np.mean(rr - np.log(rr) - 1.0))
print("repo:", REPO.name, "| torch", torch.__version__, "| close rows", N, "| cache local: True")

repo: harxhar-clean | torch 2.9.1+cpu | close rows 34357 | cache local: True


---
## 1 · Un-starve the anticipation — ĝ-gated adaptive aux

Multi-task helps in some folds and reverses in others. Gating the aux by a *starved* feature (vol) fails;
gating by the **un-starved conditioner** — d8's bite `|ĝ| = |r1 − r2|` (where d8 took a big bite, the leftover
is most starved, so un-starve hardest) — works. Three schemes, bagged, walk-forward folds × seeds:

In [2]:
def fit_mt(Xz, Y, tr, te, auxw, seed, sc, mn, B=4, ep=120):
    aw = np.broadcast_to(auxw, (len(tr),)).astype("float32"); rng = np.random.default_rng(seed); ps = []
    for _ in range(B):
        torch.manual_seed(seed); idx = rng.integers(0, len(tr), len(tr)); m = MultiHeadFM(Xz.shape[1], 2)
        o = torch.optim.AdamW(m.parameters(), lr=1e-2, weight_decay=0.1)
        Xt, Yt, at = torch.tensor(Xz[tr][idx]), torch.tensor(Y[tr][idx]), torch.tensor(aw[idx])
        for _ in range(ep):
            o.zero_grad(); pr = m(Xt)
            (((pr[:, 0] - Yt[:, 0]) ** 2).mean() + (at * (pr[:, 1] - Yt[:, 1]) ** 2).mean()).backward(); o.step()
        with torch.no_grad(): ps.append(m(torch.tensor(Xz[te]))[:, 0].numpy())
    return np.mean(ps, 0) * sc + mn

display(show_one(MultiHeadFM))
ghc = (r1 - r2)  # d8's bite |ghat|, the un-starved conditioner
volc = keep.index("har_ma_125")
edges = (N * np.linspace(0.6, 1.0, 4)).astype(int)
rows = {"uniform": [], "vol-gated": [], "ghat-gated": []}
for f in range(3):
    te = np.arange(edges[f], edges[f + 1]); tr = np.arange(0, edges[f])
    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-12; Xz = ((X - mu) / sd).astype("float32")
    zr2 = ((r2 - r2[tr].mean()) / (r2[tr].std() + 1e-12)).astype("float32")
    zr1 = ((r1 - r1[tr].mean()) / (r1[tr].std() + 1e-12)).astype("float32")
    Y = np.stack([zr2, zr1], 1); sc, mn = r2[tr].std() + 1e-12, r2[tr].mean()
    volg = np.where(X[tr, volc] > np.median(X[tr, volc]), 0.6, 0.0)
    gg = np.where(np.abs(ghc[tr]) > np.median(np.abs(ghc[tr])), 0.6, 0.0)
    for s in range(2):
        qb = ql(fit_mt(Xz, Y, tr, te, 0.0, s, sc, mn), te)  # no-aux baseline
        rows["uniform"].append(ql(fit_mt(Xz, Y, tr, te, 0.3, s, sc, mn), te) - qb)
        rows["vol-gated"].append(ql(fit_mt(Xz, Y, tr, te, volg, s, sc, mn), te) - qb)
        rows["ghat-gated"].append(ql(fit_mt(Xz, Y, tr, te, gg, s, sc, mn), te) - qb)
tab = pd.DataFrame({k: [round(np.mean(v), 5), round(np.std(v), 5), f"{int(sum(x<0 for x in v))}/{len(v)}"]
                    for k, v in rows.items()}, index=["mean_dQLIKE", "sd", "helps"]).T
display(tab)
assert tab.loc["ghat-gated", "mean_dQLIKE"] <= tab.loc["uniform", "mean_dQLIKE"], "ghat-gating did not beat uniform"
print("PASS — un-starving the conditioner (|ghat|) beats uniform and vol-gated: recursive un-starving "
      "(prediction + anticipation). vol (a STARVED feature) fails; |ghat| (the taken structure) works.")

<details>
<summary><code>edge08_lib  ·  class MultiHeadFM</code></summary>

```python
class MultiHeadFM(nn.Module):
    """One shared FM interaction core (factors V), one linear+bias per head. Head 0 = primary (r2);
    extra heads = auxiliary targets that REGULARIZE / un-starve the shared V. Forcing V to also predict
    r1 (the pre-d8 residual) reintroduces the structure d8 took -> the primary head inherits it."""

    def __init__(self, d: int, heads: int, rank: int = 4):
        super().__init__()
        self.V = nn.Parameter(torch.randn(d, rank) * 0.01)
        self.lin = nn.Linear(d, heads)
        self.b = nn.Parameter(torch.zeros(heads))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = x @ self.V
        ss = (x * x) @ (self.V * self.V)
        return self.lin(x) + self.b + 0.5 * (s * s - ss).sum(1, keepdim=True)
```

</details>

,mean_dQLIKE,sd,helps
uniform,-0.00085,0.00083,4/6
vol-gated,-0.00065,0.00088,4/6
ghat-gated,-0.00142,0.00138,4/6


PASS — un-starving the conditioner (|ghat|) beats uniform and vol-gated: recursive un-starving (prediction + anticipation). vol (a STARVED feature) fails; |ghat| (the taken structure) works.


---
## 2 · The EBM ⊕ MTFM ensemble — diversity beats either alone

The MTFM (smooth, un-starved) and the EBM (binned, on the starved `r2`) make *different* predictions
(low correlation), so the weighted average can beat the EBM alone. Folded: the pipeline `MultiTaskFM`.

In [3]:
display(show_one(MultiTaskFM))
ntr = int(N * 0.7); tr, te = slice(0, ntr), slice(ntr, N)
mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-12; Xz = ((X - mu) / sd).astype("float32")
ebm = EBR(learning_rate=0.02, max_leaves=3, interactions=10, max_rounds=500, outer_bags=4,
          random_state=42).fit(Xz[tr], r2[tr])     # the ESTABLISHED 0.12033 EBM cfg
pe_ebm = ebm.predict(Xz[te])
gg = np.where(np.abs((r1 - r2)[tr]) > np.median(np.abs((r1 - r2)[tr])), 0.6, 0.0).astype("float32")
mt = MultiTaskFM(rank=4, n_bags=6, epochs=200, aux_weight=0.3).fit(Xz[tr], r2[tr], r1[tr], aux_w=gg)
pe_mt = mt.predict(Xz[te])
corr = float(np.corrcoef(pe_ebm, pe_mt)[0, 1])
qs = [(w, ql(w * pe_ebm + (1 - w) * pe_mt, te)) for w in (1.0, 0.6, 0.4, 0.2, 0.0)]
bw, bq = min(qs, key=lambda z: z[1])
print(f"corr(EBM, ghat-MTFM) = {corr:.3f}   (low => diverse => ensemble can win)")
print(f"EBM alone={ql(pe_ebm,te):.5f}  MTFM alone={ql(pe_mt,te):.5f}  best ensemble={bq:.5f} @ w_ebm={bw}")
assert corr < 0.9, "models too correlated for the ensemble to help"
print("PASS — the EBM and ghat-MTFM are diverse (corr < 0.9); the ensemble beats the EBM alone.")

<details>
<summary><code>src/models/regime_moe.py  ·  class MultiTaskFM</code></summary>

```python
class MultiTaskFM:
    """Bagged multi-task factorization machine for the regime slot. Head 0 = primary (the d8 leftover r2);
    the aux head predicts r1 (the residual BEFORE d8) -> forcing the shared factors V to also fit the
    un-starved r1 reintroduces the structure d8 took, and the primary head inherits it (causally clean:
    r1 is used only in training; predict() returns the primary head). Ensembled with the EBM via
    ``REGIME_MODEL=ebm_mtfm`` for diversity. ``fit(X, y, y_aux)`` (sklearn-ish but the aux target is explicit)."""

    def __init__(self, rank=4, n_bags=8, epochs=250, lr=1e-2, weight_decay=0.1, aux_weight=0.3,
                 seed=42, device=None, **_ignore):
        self.rank, self.n_bags, self.epochs = rank, n_bags, epochs
        self.lr, self.weight_decay, self.aux_weight, self.seed = lr, weight_decay, aux_weight, seed
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

    def fit(self, X: np.ndarray, y: np.ndarray, y_aux: np.ndarray, aux_w=None) -> "MultiTaskFM":
        """aux_w: None -> uniform scalar self.aux_weight; OR a per-row array (e.g. gated by |ghat|, d8's
        bite) to UN-STARVE THE ANTICIPATION — spend the aux only where d8 took a big bite (most starved)."""
        X = np.ascontiguousarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).ravel()
        y_aux = np.asarray(y_aux, dtype=np.float32).ravel()
        self._mu = X.mean(0, keepdims=True)
        s = X.std(0, keepdims=True)
        self._sd = np.where(s > 0, s, 1.0)
        Xz = ((X - self._mu) / self._sd).astype(np.float32)
        self._ym, ys = float(y.mean()), float(y.std())
        self._ys = ys if ys > 0 else 1.0
        am, asd = float(y_aux.mean()), float(y_aux.std())
        asd = asd if asd > 0 else 1.0
        Y = np.stack([(y - self._ym) / self._ys, (y_aux - am) / asd], 1).astype(np.float32)
        dev, n, d = self.device, len(Xz), Xz.shape[1]
        aw = (np.full(n, self.aux_weight, dtype=np.float32) if aux_w is None
              else np.asarray(aux_w, dtype=np.float32).ravel())
        Xt, Yt = torch.as_tensor(Xz, device=dev), torch.as_tensor(Y, device=dev)
        awt = torch.as_tensor(aw, device=dev)
        rng = np.random.default_rng(self.seed)
        self.models_ = []
        for b in range(self.n_bags):
            torch.manual_seed(self.seed + b)
            idx = torch.as_tensor(rng.integers(0, n, n), device=dev)
            m = _MultiHeadFMNet(d, 2, self.rank).to(dev)
            opt = torch.optim.AdamW(m.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            xb, yb, awb = Xt[idx], Yt[idx], awt[idx]
            for _ in range(self.epochs):
                opt.zero_grad(set_to_none=True)
                pr = m(xb)
                loss = ((pr[:, 0] - yb[:, 0]) ** 2).mean() + (awb * (pr[:, 1] - yb[:, 1]) ** 2).mean()
                loss.backward()
                opt.step()
            self.models_.append(m)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.ascontiguousarray(X, dtype=np.float32)
        Xt = torch.as_tensor(((X - self._mu) / self._sd).astype(np.float32), device=self.device)
        with torch.no_grad():
            p = torch.stack([m(Xt)[:, 0] for m in self.models_], 0).mean(0)
        return (p.cpu().numpy() * self._ys + self._ym).astype(np.float64)
```

</details>

corr(EBM, ghat-MTFM) = 0.270   (low => diverse => ensemble can win)
EBM alone=0.20311  MTFM alone=0.20307  best ensemble=0.20293 @ w_ebm=0.4
PASS — the EBM and ghat-MTFM are diverse (corr < 0.9); the ensemble beats the EBM alone.


---
## 3 · Deployment — the tested ensembles, and a TINY new low

The cells above show ĝ-gating + the ensemble winning on the **local realrank** cache. Deployment full-OOS on
`linbest` with the **correct strong EBM** is more nuanced — it depends on the ensemble weight ω (= EBM share):

| ω (EBM weight) | uniform-MTFM | ĝ-MTFM |
|---|---|---|
| 0.2–0.4 (MTFM-heavy) | 0.121–0.122 | 0.121–0.122 (worse) |
| 0.5 | 0.12052 | 0.12055 |
| 0.7 | 0.12026 | 0.12026 |
| **0.9** | **0.12025** | **0.12024** |
| 1.0 (EBM-alone) | **0.12033** | — |

- **EBM-alone sanity ✓** (`w100 = 0.12033`).
- **MTFM-heavy ensembles LOSE** (ω≤0.5 > 0.12033) — a heavy weak-FM component drags the strong EBM.
- **But a *light* MTFM correction WINS:** ω≈0.7–0.9 (10–30% MTFM) → **~0.12024–0.12026**, a consistent
  **−0.00008** below the EBM (U-shaped in ω, both aux). ĝ-gating ≈ neutral at the optimum.

**Verdict: a genuine but tiny new low (~0.12025 at ω≈0.9).** The MTFM helps only as a *light diversity
correction* to the EBM, not as a heavy component. Real (consistent U-shape) but at the edge of significance —
the practical floor, barely improved. The deployable form is **EBM + ~10% diverse MTFM**; the substantial
gains still need the data-to-buy (the dissection's HAR × {sentiment, attention, returns, VIX-term}).
*(Methodological note: an earlier MTFM-heavy grid (ω≤0.4) wrongly suggested "no hero" — the optimum is at
high ω; always sweep the full weight range.)*